# 🚀 Milestone 5 — Fine-Tuning Pretrained Audio Transformer (AST)
**This is your key to hitting 0.80+ Macro F1**

### What is AST?
- **Audio Spectrogram Transformer** — pretrained on AudioSet (2M audio clips)
- Already knows what drums, bass, vocals, harmony sound like
- We just teach it the 10 genre labels = **fine-tuning**
- Think of it like hiring an expert musician and giving them a short test

### Why this works:
- ImageNet pretrained CNNs work well on new vision tasks
- AudioSet pretrained transformers work well on new audio tasks
- Much less data needed, much faster convergence

In [ ]:
!pip install transformers librosa wandb -q

In [ ]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
if DEVICE.type != 'cuda':
    print('⚠️  GPU not found! AST fine-tuning will be SLOW without GPU.')
    print('   Enable GPU: Settings → Accelerator → GPU T4 x2')

BASE        = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR   = f'{BASE}/genres_stems'
MASHUPS_DIR = f'{BASE}/mashups'
TEST_CSV    = f'{BASE}/test.csv'
GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
ROLL_NO     = 'YOUR_ROLL_NO'  # ⚠️ change this!
USE_STEMS   = ['vocals','drums','bass']   # use 3 stems

## 1️⃣ Load Pretrained AST Feature Extractor
The `ASTFeatureExtractor` handles all audio → model input conversion for us.

In [ ]:
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
print(f'Loading feature extractor from {MODEL_NAME}...')
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('✅ Feature extractor loaded')
print(f'   Sample rate : {feature_extractor.sampling_rate}')
print(f'   Max length  : {feature_extractor.max_length}')

In [ ]:
CONFIG = {
    'model_name'  : MODEL_NAME,
    'batch_size'  : 16,         # smaller because AST is large
    'epochs'      : 20,
    'lr'          : 1e-4,       # small LR for fine-tuning
    'weight_decay': 1e-4,
    'warmup_steps': 100,
    'duration'    : 30,
    'use_stems'   : USE_STEMS,
    'model'       : 'AST_finetuned',
}

TARGET_SR = feature_extractor.sampling_rate
print(f'Target sample rate: {TARGET_SR} Hz')

## 2️⃣ Dataset with AST Preprocessing

In [ ]:
class ASTDataset(Dataset):
    """
    Loads audio files and processes them using the AST feature extractor.
    Returns input_values (mel-filterbank features) ready for the transformer.
    """
    def __init__(self, file_list, label_list=None, augment=False):
        self.files   = file_list
        self.labels  = label_list
        self.augment = augment

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            y, sr = librosa.load(path, sr=TARGET_SR, duration=CONFIG['duration'])
            if len(y) < TARGET_SR:
                y = np.zeros(TARGET_SR * CONFIG['duration'])

            # Optional: simple noise augmentation
            if self.augment and random.random() < 0.3:
                noise = np.random.randn(len(y)) * 0.005
                y = y + noise

        except:
            y = np.zeros(TARGET_SR * CONFIG['duration'])

        # Feature extractor returns a dict with 'input_values'
        inputs = feature_extractor(
            y,
            sampling_rate=TARGET_SR,
            return_tensors='pt',
            padding='max_length',
            max_length=feature_extractor.max_length,
        )
        item = {'input_values': inputs['input_values'].squeeze(0)}  # (1024, 128)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Build file lists
all_files, all_labels = [], []
for genre in GENRES:
    for song in sorted(os.listdir(f'{STEMS_DIR}/{genre}')):
        for stem in USE_STEMS:
            path = f'{STEMS_DIR}/{genre}/{song}/{stem}.wav'
            if os.path.exists(path):
                all_files.append(path)
                all_labels.append(genre)

le = LabelEncoder()
labels_enc = le.fit_transform(all_labels)

X_tr, X_val, y_tr, y_val = train_test_split(
    all_files, labels_enc, test_size=0.2, stratify=labels_enc, random_state=42)

print(f'Train: {len(X_tr)}, Val: {len(X_val)}')

train_loader = DataLoader(ASTDataset(X_tr, y_tr, augment=True),
                          batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader   = DataLoader(ASTDataset(X_val, y_val, augment=False),
                          batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

## 3️⃣ Load & Modify Pretrained AST
AST was pretrained to classify 527 AudioSet classes.
We replace its output head with a 10-class head for our genres.

In [ ]:
# Create label mappings (required by HuggingFace)
id2label = {i: g for i, g in enumerate(le.classes_)}
label2id = {g: i for i, g in id2label.items()}

print('Loading pretrained AST model...')
model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels      = len(GENRES),
    id2label        = id2label,
    label2id        = label2id,
    ignore_mismatched_sizes = True,  # replaces the 527-class head with 10-class
)
model = model.to(DEVICE)
print('✅ AST model loaded and adapted for 10 genres')

total_params  = sum(p.numel() for p in model.parameters())
train_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {train_params:,}')

In [ ]:
# OPTIONAL: Freeze most of the model and only train the last few layers + head
# This is faster and prevents overfitting on small datasets.
# Comment this out to fine-tune the full model (slower but potentially better).

# Freeze all except last 2 transformer blocks and classifier head
for name, param in model.named_parameters():
    param.requires_grad = False

# Unfreeze last 2 encoder layers
for name, param in model.named_parameters():
    if any(f'encoder.layer.{i}' in name for i in [10, 11]) or 'classifier' in name:
        param.requires_grad = True

train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Fine-tuning {train_params:,} params (last 2 layers + head)')
print('To unfreeze all: comment out the freeze block above')

## 4️⃣ Training

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        input_vals = batch['input_values'].to(DEVICE)
        y          = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(input_values=input_vals)
        loss    = criterion(outputs.logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item() * len(y)
        preds.extend(outputs.logits.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return total_loss/len(loader.dataset), f1_score(labels, preds, average='macro')

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        input_vals = batch['input_values'].to(DEVICE)
        y          = batch['labels'].to(DEVICE)
        outputs    = model(input_values=input_vals)
        loss       = criterion(outputs.logits, y)
        total_loss += loss.item() * len(y)
        preds.extend(outputs.logits.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    return total_loss/len(loader.dataset), f1_score(labels, preds, average='macro'), preds, labels

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

wandb.init(project=f'{ROLL_NO}-t12026', name='milestone5-ast-finetune', config=CONFIG)

best_val_f1 = 0
train_h = {'loss':[],'f1':[]}; val_h = {'loss':[],'f1':[]}
print(f'Fine-tuning AST for {CONFIG["epochs"]} epochs...')

In [ ]:
for epoch in range(1, CONFIG['epochs']+1):
    t0 = time.time()
    tr_loss, tr_f1 = train_epoch(model, train_loader, optimizer, None, criterion)
    va_loss, va_f1, va_preds, va_labels = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    train_h['loss'].append(tr_loss); train_h['f1'].append(tr_f1)
    val_h['loss'].append(va_loss);   val_h['f1'].append(va_f1)

    if va_f1 > best_val_f1:
        best_val_f1 = va_f1
        model.save_pretrained('best_ast')
        torch.save(model.state_dict(), 'best_ast.pt')
        print(f'  💾 Best saved! F1={va_f1:.4f}')

    wandb.log({'epoch': epoch, 'train_loss': tr_loss, 'train_f1': tr_f1,
               'val_loss': va_loss, 'val_f1': va_f1})

    print(f'Epoch {epoch:2d}/{CONFIG["epochs"]} | '
          f'Train F1: {tr_f1:.4f} | Val F1: {va_f1:.4f} | '
          f'{time.time()-t0:.1f}s')

print(f'\n🎯 Best Val Macro F1: {best_val_f1:.4f}')
print('Target is 0.80 — submit to Kaggle to get your official score!')

In [ ]:
# Curves
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for i,(k,title) in enumerate([('loss','Loss'),('f1','Macro F1')]):
    axes[i].plot(train_h[k], label='Train', color='steelblue')
    axes[i].plot(val_h[k],   label='Val',   color='tomato')
    if k=='f1': axes[i].axhline(0.80, color='green', linestyle='--', label='Target 0.80')
    axes[i].set_title(f'AST Fine-tune — {title}'); axes[i].legend()
plt.tight_layout(); plt.savefig('ast_curves.png', dpi=150); plt.show()
wandb.log({'ast_curves': wandb.Image('ast_curves.png')})

# Best model report
model.load_state_dict(torch.load('best_ast.pt'))
_, best_f1, best_preds, best_labels = eval_epoch(model, val_loader, criterion)
print(f'\nBest AST Val Macro F1: {best_f1:.4f}')
print(classification_report(best_labels, best_preds, target_names=le.classes_))
wandb.log({'best_val_macro_f1': best_f1})
wandb.finish()

## 5️⃣ Kaggle Submission

In [ ]:
test_df   = pd.read_csv(TEST_CSV)
fname_col = test_df.columns[1]
test_files= [f'{MASHUPS_DIR}/{row[fname_col]}' for _, row in test_df.iterrows()]
test_ds   = ASTDataset(test_files)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'],
                         shuffle=False, num_workers=2)

model.eval()
all_preds = []
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        inp  = batch['input_values'].to(DEVICE)
        out  = model(input_values=inp)
        all_preds.extend(out.logits.argmax(1).cpu().numpy())
        if (i+1) % 20 == 0: print(f'  {(i+1)*CONFIG["batch_size"]}/{len(test_df)} done')

submission = pd.DataFrame({'id': test_df['id'], 'genre': le.inverse_transform(all_preds)})
submission.to_csv('submission_m5_ast.csv', index=False)
print('✅ submission_m5_ast.csv saved')
print(submission['genre'].value_counts())

## 6️⃣ Tips if Score < 0.80

Try these in order:
1. **Unfreeze full model** — remove the freeze block, fine-tune all layers
2. **Use all 4 stems** — add 'others' to USE_STEMS
3. **More epochs** — set epochs=30
4. **Lower learning rate** — try lr=5e-5
5. **Test-time augmentation (TTA)** — predict multiple crops, take majority vote
6. **Ensemble** — average predictions from CNN + CRNN + AST